<a href="https://colab.research.google.com/github/txellbalada/Reto_IA/blob/main/500Datos_Reales_colomb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Reto IA: Telefonica challenge:
###Objetivo: entrenar un modelo capaz de distinguir entre un áudio real vs uno generado con IA

In [ ]:
import numpy as np
import pandas as pd
import librosa
import os
from scipy import stats
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%ls /content/drive/MyDrive/Reto_Telefonica/Dataset_real/Acento_Colombiano_real_FM/

cof_00610_00008989777.wav  cof_01523_00241535803.wav  com_00610_01226934508.wav
cof_00610_00011481761.wav  cof_01523_00263223799.wav  com_00610_01229503605.wav
cof_00610_00026180919.wav  cof_01523_00272438245.wav  com_00610_01236993729.wav
cof_00610_00083325222.wav  cof_01523_00280869759.wav  com_00610_01242095809.wav
cof_00610_00100787111.wav  cof_01523_00286325544.wav  com_00610_01332319294.wav
cof_00610_00106068443.wav  cof_01523_00287301417.wav  com_00610_01345253353.wav
cof_00610_00128829414.wav  cof_01523_00295602931.wav  com_00610_01347530709.wav
cof_00610_00129157870.wav  cof_01523_00300070090.wav  com_00610_01356460222.wav
cof_00610_00131238701.wav  cof_01523_00310910177.wav  com_00610_01357734740.wav
cof_00610_00132949807.wav  cof_01523_00324308563.wav  com_00610_01399950713.wav
cof_00610_00137907909.wav  cof_01523_00330363745.wav  com_00610_01412052726.wav
cof_00610_00143476359.wav  cof_01523_00354041865.wav  com_00610_01427091610.wav
cof_00610_00144757091.wav  cof_01523_003

In [ ]:
TIPO_DATASET = "real"
ruta_carpeta = "/content/drive/MyDrive/Reto_Telefonica/Dataset_real/Acento_Colombiano_real_FM/"

In [ ]:
print("Carpeta configurada:")
print(ruta_carpeta)

print("\n¿Existe la carpeta?:", os.path.exists(ruta_carpeta))

if os.path.exists(ruta_carpeta):
    print("\nPrimeros archivos encontrados:")
    print(os.listdir(ruta_carpeta)[:10])

Carpeta configurada:
/content/drive/MyDrive/Reto_Telefonica/Dataset_real/Acento_Colombiano_real_FM/

¿Existe la carpeta?: True

Primeros archivos encontrados:
['cof_00610_00756082844.wav', 'cof_00610_01570108574.wav', 'cof_00610_01122218651.wav', 'cof_00610_00798094083.wav', 'cof_00610_01768560254.wav', 'cof_00610_00551983037.wav', 'cof_00610_01352413514.wav', 'cof_00610_00083325222.wav', 'cof_00610_01967418412.wav', 'cof_00610_00215905338.wav']


In [ ]:
#Cargar audios
archivos = os.listdir(ruta_carpeta)
print(f"Se detectaron {len(archivos)} archivos en la carpeta.\n")

for archivo in archivos:
    if archivo.lower().endswith(".wav"):
        ruta_audio = os.path.join(ruta_carpeta, archivo)

        y, sr = librosa.load(ruta_audio, sr=16000)

        print(f"{archivo} cargado | duración: {len(y)/sr:.2f} segundos")

Se detectaron 500 archivos en la carpeta.

cof_00610_00756082844.wav cargado | duración: 5.38 segundos
cof_00610_01570108574.wav cargado | duración: 5.89 segundos
cof_00610_01122218651.wav cargado | duración: 3.67 segundos
cof_00610_00798094083.wav cargado | duración: 5.97 segundos
cof_00610_01768560254.wav cargado | duración: 3.93 segundos
cof_00610_00551983037.wav cargado | duración: 4.95 segundos
cof_00610_01352413514.wav cargado | duración: 6.40 segundos
cof_00610_00083325222.wav cargado | duración: 5.89 segundos
cof_00610_01967418412.wav cargado | duración: 3.75 segundos
cof_00610_00215905338.wav cargado | duración: 5.29 segundos
cof_00610_01067172216.wav cargado | duración: 5.21 segundos
cof_00610_01803432611.wav cargado | duración: 4.44 segundos
cof_00610_00913180829.wav cargado | duración: 4.01 segundos
cof_00610_00864540466.wav cargado | duración: 5.72 segundos
cof_00610_00648896188.wav cargado | duración: 6.23 segundos
cof_00610_00797573887.wav cargado | duración: 3.84 segund

In [ ]:
def extraer_etiquetas(nombre_audio, modo="auto"):
    """
    Extrae etiquetas binarias desde el nombre del audio.

    Devuelve un diccionario con:
    - label: 0 real / 1 sintético
    - genero_f: 1 mujer / 0 hombre
    - nacionalidad: colombiano / chileno / argentino
    - modelo generador: CycleGAN, Diff, StarGAN, TTS-Dif, TTS-StarGAN, TTS
    """

    nombre = nombre_audio.lower()

    etiquetas = {
        # Label principal
        "label": 0,

        # Género (solo una columna)
        "genero_f": 0,

        # Nacionalidad
        "colombiano": 0,
        "chileno": 0,
        "argentino": 0,

        # Modelo generador
        "modelo_cyclegan": 0,
        "modelo_diff": 0,
        "modelo_stargan": 0,
        "modelo_tts_dif": 0,
        "modelo_tts_stargan": 0,
        "modelo_tts": 0
    }

    # -----------------------------
    # 1. Label principal
    # -----------------------------
    if modo == "real":
        etiquetas["label"] = 0
    elif modo == "sintetico":
        etiquetas["label"] = 1
    else:
        etiquetas["label"] = 1 if "-" in nombre_audio else 0

    # -----------------------------
    # 2. Nacionalidad + género
    # -----------------------------
    if "com" in nombre:
        etiquetas["colombiano"] = 1
        etiquetas["genero_f"] = 0

    elif "cof" in nombre:
        etiquetas["colombiano"] = 1
        etiquetas["genero_f"] = 1

    elif "clm" in nombre:
        etiquetas["chileno"] = 1
        etiquetas["genero_f"] = 0

    elif "clf" in nombre:
        etiquetas["chileno"] = 1
        etiquetas["genero_f"] = 1

    elif "arm" in nombre:
        etiquetas["argentino"] = 1
        etiquetas["genero_f"] = 0

    elif "arf" in nombre:
        etiquetas["argentino"] = 1
        etiquetas["genero_f"] = 1

    # -----------------------------
    # 3. Modelo generador
    # -----------------------------
    nombre_original = nombre_audio  # mantener mayúsculas

    if "TTS-StarGAN" in nombre_original:
        etiquetas["modelo_tts_stargan"] = 1
    elif "TTS-Dif" in nombre_original:
        etiquetas["modelo_tts_dif"] = 1
    elif "CycleGAN" in nombre_original:
        etiquetas["modelo_cyclegan"] = 1
    elif "StarGAN" in nombre_original:
        etiquetas["modelo_stargan"] = 1
    elif "Diff" in nombre_original:
        etiquetas["modelo_diff"] = 1
    elif "TTS" in nombre_original:
        etiquetas["modelo_tts"] = 1

    return etiquetas

In [ ]:
def generar_df_etiquetas(ruta_carpeta, tipo_dataset="auto"):
    """
    Genera un DataFrame con múltiples etiquetas:
    - id_audio: identificador secuencial (1, 2, 3, ...)
    - archivo: nombre original del archivo
    - label: real (0) vs sintético (1)
    - genero_f: mujer (1), hombre (0)
    - nacionalidad: colombiano / chileno / argentino
    - modelo generador
    """

    print(f"Explorando carpeta: {ruta_carpeta}")
    datos = []

    contador_id = 1

    for archivo in sorted(os.listdir(ruta_carpeta)):
        if archivo.lower().endswith(".wav"):
            nombre_audio = archivo.replace(".wav", "")

            etiquetas = extraer_etiquetas(nombre_audio, modo=tipo_dataset)

            fila = {
                "id_audio": contador_id,
                "archivo": archivo
            }

            fila.update(etiquetas)
            datos.append(fila)

            contador_id += 1

    df_etiquetas = pd.DataFrame(datos)

    print(f"DataFrame creado con {len(df_etiquetas)} registros.")
    print(f"Número de columnas: {df_etiquetas.shape[1]}")

    return df_etiquetas

In [ ]:
df_labels = generar_df_etiquetas(ruta_carpeta, tipo_dataset=TIPO_DATASET)
df_labels.head()

Explorando carpeta: /content/drive/MyDrive/Reto_Telefonica/Dataset_real/Acento_Colombiano_real_FM/
DataFrame creado con 500 registros.
Número de columnas: 13


,id_audio,archivo,label,genero_f,colombiano,chileno,argentino,modelo_cyclegan,modelo_diff,modelo_stargan,modelo_tts_dif,modelo_tts_stargan,modelo_tts
0,1,cof_00610_00008989777.wav,0,1,1,0,0,0,0,0,0,0,0
1,2,cof_00610_00011481761.wav,0,1,1,0,0,0,0,0,0,0,0
2,3,cof_00610_00026180919.wav,0,1,1,0,0,0,0,0,0,0,0
3,4,cof_00610_00083325222.wav,0,1,1,0,0,0,0,0,0,0,0
4,5,cof_00610_00100787111.wav,0,1,1,0,0,0,0,0,0,0,0


##  Mejor opción para TU problema (detección de voz fake)

Para detectar audio sintético, lo más importante es capturar:

- textura espectral (cómo suena la voz)  
- irregularidades (artefactos de IA)  
- dinámica (energía, variaciones)  

 Por eso, la mejor combinación es:


###  1. MFCC + Delta MFCC (OBLIGATORIO)

Son las más importantes.

Capturan:

- forma del espectro (timbre)  
- cambios en el tiempo  

✔ Detectan muy bien voces sintéticas  


###  2. Features de energía y estructura

- RMS → energía  
- ZCR → ruido/aspereza  

✔ Útiles para detectar artefactos de IA  



###  3. Features espectrales

- Spectral centroid → brillo  
- Bandwidth → dispersión  
- Rolloff → límite de energía  
- Flatness → tonal vs ruido  

✔ Clave para distinguir natural vs artificial  


###  4. Spectral contrast (MUY importante)

Esto muchas veces se subestima, pero:

 captura diferencias entre bandas de frecuencia  

✔ Muy útil para detectar:

- vocoders  
- modelos tipo GAN / TTS  


###  5. Estadísticas (CRÍTICO)

Esto es lo que mucha gente hace mal:

 NO usar los valores frame a frame  
 SÍ usar resumen:

- mean  
- std  
- min  
- max  
- median  
- q1  
- q3  

✔ Esto convierte el audio en una representación robusta  


## Entonces… ¿cuál es la mejor opción?

 EXACTAMENTE la que implementaste en el notebook nuevo:

✔ MFCC (13)  
✔ Delta MFCC  
✔ ZCR  
✔ RMS  
✔ Spectral features  
✔ Spectral contrast  
✔ Estadísticas completas  

In [ ]:
def resumir_feature(vector, prefijo, debug=False):
    """
    Resume un vector numérico con estadísticas descriptivas avanzadas.
    """
    vector = np.asarray(vector).astype(float)

    if len(vector) == 0:
        return {}

    resultado = {
        f"{prefijo}_mean": np.mean(vector),
        f"{prefijo}_std": np.std(vector),
        f"{prefijo}_min": np.min(vector),
        f"{prefijo}_max": np.max(vector),
        f"{prefijo}_median": np.median(vector),
        f"{prefijo}_q1": np.quantile(vector, 0.25),
        f"{prefijo}_q3": np.quantile(vector, 0.75),
        f"{prefijo}_skew": stats.skew(vector),
        f"{prefijo}_kurtosis": stats.kurtosis(vector),
        f"{prefijo}_mode": stats.mode(vector, keepdims=True)[0][0],
        f"{prefijo}_iqr": stats.iqr(vector)
    }

    if debug:
        print(f"\n🔍 Feature: {prefijo}")
        print(f"   Total valores generados: {len(resultado)}")
        for k, v in list(resultado.items())[:3]:
            print(f"   {k}: {v:.4f}")

    return resultado

In [ ]:
def extraer_features(ruta_audio, sr_objetivo=16000, n_mfcc=13):
    """
    Extrae features de un audio y devuelve un diccionario.
    """

    y, sr = librosa.load(ruta_audio, sr=sr_objetivo)

    # Recorte de silencios
    y, _ = librosa.effects.trim(y)

    if len(y) < 512:
        return None

    features = {}

    # Meta
    features["duracion_seg"] = len(y) / sr

    # ZCR
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.update(resumir_feature(zcr, "zcr"))

    # RMS
    rms = librosa.feature.rms(y=y)[0]
    features.update(resumir_feature(rms, "rms"))

    # RMSE manual
    rmse_manual = np.sqrt(np.mean(y**2))
    features["rmse_manual"] = rmse_manual

    # Tempo
    tempo = librosa.beat.tempo(y=y, sr=sr)[0]
    features["tempo"] = tempo

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    for i in range(n_mfcc):
        features.update(resumir_feature(mfcc[i], f"mfcc_{i+1}"))

    # Delta MFCC
    delta_mfcc = librosa.feature.delta(mfcc)
    for i in range(n_mfcc):
        features.update(resumir_feature(delta_mfcc[i], f"delta_mfcc_{i+1}"))

    # Delta-Delta MFCC
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    for i in range(n_mfcc):
        features.update(resumir_feature(delta2_mfcc[i], f"delta2_mfcc_{i+1}"))

    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features.update(resumir_feature(centroid, "centroid"))

    # Spectral bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features.update(resumir_feature(bandwidth, "bandwidth"))

    # Spectral contrast
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    for i in range(contrast.shape[0]):
        features.update(resumir_feature(contrast[i], f"contrast_{i+1}"))

    # Spectral flatness
    flatness = librosa.feature.spectral_flatness(y=y)[0]
    features.update(resumir_feature(flatness, "flatness"))

    # Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features.update(resumir_feature(rolloff, "rolloff"))

    print(f"🎧 {os.path.basename(ruta_audio)}")
    print(f"   - Features extraídas: {len(features)}")
    print(f"   - Tempo: {tempo:.2f}")
    print(f"   - RMSE manual: {rmse_manual:.5f}")

    return features

In [ ]:
print("\n Probando extracción de features con un audio...\n")

for archivo in sorted(os.listdir(ruta_carpeta)):
    if archivo.lower().endswith(".wav"):
        ruta_audio_prueba = os.path.join(ruta_carpeta, archivo)

        try:
            ejemplo_features = extraer_features(ruta_audio_prueba)

            if ejemplo_features is None:
                print(f" Audio omitido (muy corto): {archivo}")
                continue

            print("\n Resultado:")
            print(f"   - Audio: {archivo}")
            print(f"   - Nº features: {len(ejemplo_features)}")

            print("\n Ejemplo de features:")
            for k, v in list(ejemplo_features.items())[:5]:
                print(f"   {k}: {v}")

            valores = list(ejemplo_features.values())
            n_nan = sum(np.isnan(valores))
            n_inf = sum(np.isinf(valores))

            print("\n Control de calidad:")
            print(f"   - NaN: {n_nan}")
            print(f"   - Inf: {n_inf}")

            break

        except Exception as e:
            print(f" Error con {archivo}: {e}")
            continue


 Probando extracción de features con un audio...



/tmp/ipykernel_9082/678304563.py:32: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]


🎧 cof_00610_00008989777.wav
   - Features extraídas: 575
   - Tempo: 170.45
   - RMSE manual: 0.05312

 Resultado:
   - Audio: cof_00610_00008989777.wav
   - Nº features: 575

 Ejemplo de features:
   duracion_seg: 4.5333125
   zcr_mean: 0.12801909110915494
   zcr_std: 0.10808946207186829
   zcr_min: 0.01123046875
   zcr_max: 0.52099609375

 Control de calidad:
   - NaN: 0
   - Inf: 0


In [ ]:
registros = []

archivos_wav = sorted([a for a in os.listdir(ruta_carpeta) if a.lower().endswith(".wav")])

for idx, archivo in enumerate(tqdm(archivos_wav), start=1):
    ruta_audio = os.path.join(ruta_carpeta, archivo)
    nombre_audio = archivo.replace(".wav", "")

    try:
        etiquetas = extraer_etiquetas(nombre_audio, modo=TIPO_DATASET)
        feats = extraer_features(ruta_audio)

        if feats is None:
            print(f"Audio omitido por ser demasiado corto: {archivo}")
            continue

        fila = {
            "id_audio": idx,
            "archivo": archivo
        }

        fila.update(etiquetas)
        fila.update(feats)

        registros.append(fila)

    except Exception as e:
        print(f"Error procesando {archivo}: {e}")

  0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipykernel_9082/678304563.py:32: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]
  0%|          | 1/500 [00:00<01:48,  4.61it/s]

🎧 cof_00610_00008989777.wav
   - Features extraídas: 575
   - Tempo: 170.45
   - RMSE manual: 0.05312


  0%|          | 2/500 [00:00<01:47,  4.63it/s]

🎧 cof_00610_00011481761.wav
   - Features extraídas: 575
   - Tempo: 156.25
   - RMSE manual: 0.05705


  1%|          | 3/500 [00:00<01:48,  4.56it/s]

...

🎧 com_01523_01286140247.wav
   - Features extraídas: 575
   - Tempo: 89.29
   - RMSE manual: 0.07126


100%|█████████▉| 499/500 [02:10<00:00,  3.10it/s]

🎧 com_01523_01299556545.wav
   - Features extraídas: 575
   - Tempo: 125.00
   - RMSE manual: 0.08986


100%|██████████| 500/500 [02:11<00:00,  3.81it/s]

🎧 com_01523_01351282485.wav
   - Features extraídas: 575
   - Tempo: 110.29
   - RMSE manual: 0.08377


In [ ]:
print("\n Creando DataFrame...")

df = pd.DataFrame(registros)

print(" DataFrame creado")
print(f"   - Filas: {df.shape[0]}")
print(f"   - Columnas: {df.shape[1]}")

df.head()


 Creando DataFrame...
 DataFrame creado
   - Filas: 500
   - Columnas: 588


,id_audio,archivo,label,genero_f,colombiano,chileno,argentino,modelo_cyclegan,modelo_diff,modelo_stargan,...,rolloff_std,rolloff_min,rolloff_max,rolloff_median,rolloff_q1,rolloff_q3,rolloff_skew,rolloff_kurtosis,rolloff_mode,rolloff_iqr
0,1,cof_00610_00008989777.wav,0,1,1,0,0,0,0,0,...,2052.610133,148.4375,7046.8750,4171.87500,2703.125000,5951.171875,-0.464093,-1.007046,187.5000,3248.046875
1,2,cof_00610_00011481761.wav,0,1,1,0,0,0,0,0,...,1802.043666,351.5625,7226.5625,3691.40625,2548.828125,5773.437500,0.028474,-1.184660,1226.5625,3224.609375
2,3,cof_00610_00026180919.wav,0,1,1,0,0,0,0,0,...,2068.538695,148.4375,7195.3125,3437.50000,2603.515625,6167.968750,-0.163310,-1.342991,2734.3750,3564.453125
3,4,cof_00610_00083325222.wav,0,1,1,0,0,0,0,0,...,1927.039568,148.4375,7375.0000,3062.50000,2429.687500,5933.593750,0.229813,-1.189844,2679.6875,3503.906250
4,5,cof_00610_00100787111.wav,0,1,1,0,0,0,0,0,...,1914.970894,312.5000,7414.0625,4539.06250,2552.734375,6128.906250,-0.241595,-1.345283,6039.0625,3576.171875


In [ ]:
print("\n Distribución de labels:")
print(df["label"].value_counts())

print("\nProporciones:")
print(df["label"].value_counts(normalize=True))


 Distribución de labels:
label
0    500
Name: count, dtype: int64

Proporciones:
label
0    1.0
Name: proportion, dtype: float64


In [ ]:
#Revisar columnas
print("\n Información de columnas:")

print(f"Total columnas: {len(df.columns)}")

print("\nPrimeras 20 columnas:")
print(df.columns[:20])

print("\nÚltimas 20 columnas:")
print(df.columns[-20:])


 Información de columnas:
Total columnas: 588

Primeras 20 columnas:
Index(['id_audio', 'archivo', 'label', 'genero_f', 'colombiano', 'chileno',
       'argentino', 'modelo_cyclegan', 'modelo_diff', 'modelo_stargan',
       'modelo_tts_dif', 'modelo_tts_stargan', 'modelo_tts', 'duracion_seg',
       'zcr_mean', 'zcr_std', 'zcr_min', 'zcr_max', 'zcr_median', 'zcr_q1'],
      dtype='object')

Últimas 20 columnas:
Index(['flatness_min', 'flatness_max', 'flatness_median', 'flatness_q1',
       'flatness_q3', 'flatness_skew', 'flatness_kurtosis', 'flatness_mode',
       'flatness_iqr', 'rolloff_mean', 'rolloff_std', 'rolloff_min',
       'rolloff_max', 'rolloff_median', 'rolloff_q1', 'rolloff_q3',
       'rolloff_skew', 'rolloff_kurtosis', 'rolloff_mode', 'rolloff_iqr'],
      dtype='object')


In [ ]:
print("\n Información general del DataFrame:")
df.info()


 Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Columns: 588 entries, id_audio to rolloff_iqr
dtypes: float32(1), float64(574), int64(12), object(1)
memory usage: 2.2+ MB


In [ ]:
print("\n Estadísticas descriptivas:")
df.describe()


 Estadísticas descriptivas:


,id_audio,label,genero_f,colombiano,chileno,argentino,modelo_cyclegan,modelo_diff,modelo_stargan,modelo_tts_dif,...,rolloff_std,rolloff_min,rolloff_max,rolloff_median,rolloff_q1,rolloff_q3,rolloff_skew,rolloff_kurtosis,rolloff_mode,rolloff_iqr
count,500.000000,500.0,500.000000,500.0,500.0,500.0,500.0,500.0,500.0,500.0,...,500.000000,500.000000,500.00000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,250.500000,0.0,0.500000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1995.726868,331.484375,7105.06250,3678.117188,1996.660156,5704.640625,0.030629,-1.299474,2998.406250,3707.980469
std,144.481833,0.0,0.500501,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,158.570321,265.743904,291.84244,1107.631344,556.800159,536.364429,0.404523,0.374352,2222.253564,627.853350
min,1.000000,0.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1425.633176,109.375000,5960.93750,1484.375000,890.625000,3210.937500,-1.459108,-1.787087,117.187500,921.875000
25%,125.750000,0.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1895.188415,148.437500,6929.68750,2849.609375,1593.261719,5562.500000,-0.209840,-1.556134,1294.921875,3397.949219
50%,250.500000,0.0,0.500000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2008.559981,203.125000,7140.62500,3484.375000,1920.898438,5854.492188,0.036758,-1.396424,1875.000000,3822.265625
75%,375.250000,0.0,1.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2115.018287,414.062500,7320.31250,4438.476562,2294.921875,6053.222656,0.289497,-1.167209,5859.375000,4187.988281
max,500.000000,0.0,1.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2391.257989,1679.687500,7703.12500,6171.875000,5429.687500,6375.000000,1.167386,0.981234,6632.812500,4734.375000


In [ ]:
print("\n Inspección de una fila completa:")

fila = df.iloc[0]

print(fila)
print("\nTotal de valores en esta fila:", len(fila))


 Inspección de una fila completa:
id_audio                                    1
archivo             cof_00610_00008989777.wav
label                                       0
genero_f                                    1
colombiano                                  1
                              ...            
rolloff_q3                        5951.171875
rolloff_skew                        -0.464093
rolloff_kurtosis                    -1.007046
rolloff_mode                            187.5
rolloff_iqr                       3248.046875
Name: 0, Length: 588, dtype: object

Total de valores en esta fila: 588


In [ ]:
print("\n Guardando dataset...")

nombre_csv = f"dataset_features_{TIPO_DATASET}.csv"
ruta_salida_csv = f"/content/drive/MyDrive/Reto_Telefonica/{nombre_csv}"

df.to_csv(ruta_salida_csv, index=False)

print(" Dataset guardado correctamente")
print(f" Ruta: {ruta_salida_csv}")
print(f" Shape final: {df.shape}")


 Guardando dataset...
 Dataset guardado correctamente
 Ruta: /content/drive/MyDrive/Reto_Telefonica/dataset_features_real.csv
 Shape final: (500, 588)


In [ ]:
print("\n Preparando descarga...")

from google.colab import files
files.download(ruta_salida_csv)

print(" Descarga iniciada")


 Preparando descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Descarga iniciada


In [ ]:
print("\n Guardando archivo auxiliar de etiquetas...")

df_labels = df[["id_audio", "archivo", "label"]]

ruta_labels = f"/content/drive/MyDrive/Reto_Telefonica/labels_{TIPO_DATASET}.csv"
df_labels.to_csv(ruta_labels, index=False)

print(" Archivo de labels guardado")
print(f" Ruta: {ruta_labels}")

df_labels.head()


 Guardando archivo auxiliar de etiquetas...
 Archivo de labels guardado
 Ruta: /content/drive/MyDrive/Reto_Telefonica/labels_real.csv


,id_audio,archivo,label
0,1,cof_00610_00008989777.wav,0
1,2,cof_00610_00011481761.wav,0
2,3,cof_00610_00026180919.wav,0
3,4,cof_00610_00083325222.wav,0
4,5,cof_00610_00100787111.wav,0
